# Building a ReAct Agent in Python: Reason, Act, Observe

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/agents/react_from_scratch.ipynb)

Companion notebook to [the post](https://sesen.ai/blog/react-agent-from-scratch).

An agent is a model that can call tools, and the code that lets it do so is a loop
of about thirty lines. This notebook builds that loop, measures why interleaving
reasoning with acting beats planning everything up front, and breaks the loop three
ways before fixing each one.

**No API key is needed.** A scripted stand-in model replays a fixed transcript, so
every number here reproduces exactly and the whole notebook runs in about a second.
The last cell is optional and swaps in a real Claude call.

## 1. The tools

Two tools, both deliberately small. `search` looks up an entry by title and
suggests close titles when nothing matches, which is what the Wikipedia API did in
the original ReAct paper. `calculator` evaluates arithmetic and refuses anything
that is not arithmetic, which is why `eval` is survivable here.

In [ ]:
"""ReAct from scratch: the reason-act-observe loop, its guards, and a scripted model.

Nothing in this file calls a language model. `StubLLM` replays a fixed script keyed
on what is visible in the prompt, so every number and figure in the post regenerates
identically on any machine with no API key and no network.

The tools mirror the ones in Yao et al. (2022): a keyword `search` over a small
encyclopedia that returns suggestions when a title does not match, and a `calculator`.
"""


import re
from dataclasses import dataclass, field


class ToolError(Exception):
    """Raised when a tool is called with something it cannot use."""


ENCYCLOPEDIA = {
    "AlexNet": (
        "AlexNet is a convolutional network trained across two GPUs. It won the "
        "ImageNet competition in 2012 with a top-5 error of 15.3 percent."
    ),
    "Transformer": (
        "The Transformer was published in 2017 in the paper Attention Is All You "
        "Need. It replaced recurrence with self-attention, and the base model "
        "stacks 6 encoder layers with 8 attention heads."
    ),
    "ResNet": (
        "ResNet introduced residual connections between layers. It won the ImageNet "
        "competition in 2015, and the deepest variant in the paper has 152 layers."
    ),
    "BERT": (
        "BERT was published in 2018. It is a bidirectional encoder with 12 layers "
        "in the base configuration and 110 million parameters."
    ),
    "LSTM": (
        "The long short-term memory network is a recurrent architecture whose gated "
        "cell carries state across many time steps without the gradient vanishing. "
        "It was published in 1997 by Hochreiter and Schmidhuber."
    ),
    "Dropout": (
        "Dropout removes a random subset of activations on every training step, "
        "which stops units from co-adapting. The journal version appeared in 2014."
    ),
    "Adam": (
        "Adam is an adaptive moment optimiser that keeps running estimates of the "
        "first and second moment of the gradient and rescales each step by them. "
        "It was published in 2015."
    ),
    "GPT-3": (
        "GPT-3 was published in 2020 and has 175 billion parameters spread over "
        "96 layers, trained on roughly 300 billion tokens."
    ),
    "ReAct": (
        "ReAct interleaves reasoning traces with actions so a model can revise its "
        "plan as it reads results. It was published in 2022 by Yao and colleagues."
    ),
    "Word2Vec": (
        "Word2Vec was published in 2013. It learns word embeddings from a shallow "
        "model trained to predict the words surrounding a target word."
    ),
}

_WORD = re.compile(r"[a-z0-9]+")


def _tokens(text: str) -> set[str]:
    return set(_WORD.findall(text.lower()))


def search(query: str) -> str:
    """Look up an encyclopedia entry by title, or suggest close titles."""
    for title, body in ENCYCLOPEDIA.items():
        if title.lower() == query.strip().lower():
            return body

    query_tokens = _tokens(query)
    scored = sorted(
        ENCYCLOPEDIA,
        key=lambda t: (-len(query_tokens & _tokens(t + " " + ENCYCLOPEDIA[t])), t),
    )
    return f'Could not find "{query}". Similar: {", ".join(scored[:3])}.'


_ARITHMETIC = re.compile(r"^[\d\s+\-*/().]+$")


def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression, refusing anything that is not arithmetic."""
    if not _ARITHMETIC.match(expression):
        raise ToolError(
            f"calculator accepts digits and + - * / ( ) only, got: {expression!r}"
        )
    try:
        value = eval(expression, {"__builtins__": {}}, {})  # noqa: S307
    except Exception as exc:  # pragma: no cover - arithmetic errors only
        raise ToolError(f"calculator could not evaluate {expression!r}: {exc}") from exc
    return str(value)


TOOLS = {"search": search, "calculator": calculator}

In [ ]:
print(search("Transformer")[:90])
print(search("Attention Is All You Need"))
print(calculator("2017-1997"))
try:
    calculator("2017 minus 1997")
except ToolError as exc:
    print("refused:", exc)

## 2. The prompt

Three pieces, rebuilt from scratch on every pass: a fixed system prompt, the
question, and the scratchpad. The scratchpad is the only one that changes, and it
only ever grows.

In [ ]:
SYSTEM = """Answer the question using the tools below.

search[query]        look up an entry in the encyclopedia
calculator[expr]     evaluate an arithmetic expression

Work one step at a time in this format:

Thought: why you are doing this
Action: tool[argument]

You will then be shown the result as an Observation. When you can answer, write:

Thought: why you are done
Final Answer: the answer
"""


def build_prompt(question: str, scratchpad: str) -> str:
    return f"{SYSTEM}\nQuestion: {question}\n{scratchpad}"


ACTION = re.compile(r"^Action:\s*(\w+)\[(.*)\]\s*$", re.MULTILINE)
FINAL = re.compile(r"^Final Answer:\s*(.*)$", re.MULTILINE)

In [ ]:
print(build_prompt("How old is the LSTM paper?", ""))

## 3. A scripted stand-in for the model

`StubLLM` returns the first move whose `requires` strings are all present in the
prompt and whose action has not been taken yet. That makes the script react to
observations instead of replaying blindly: a move whose evidence never arrives
never fires, which is exactly how a real model behaves when its context is missing
the fact the answer needs.

In [ ]:
@dataclass(frozen=True)
class Move:
    """One scripted model turn.

    `requires` lists substrings that must appear in the prompt before this move is
    available. That is what makes the script react to observations rather than
    replay blindly: a move whose evidence never arrives never fires.
    """

    text: str
    requires: tuple[str, ...] = ()
    forbids: tuple[str, ...] = ()

    @property
    def action_line(self) -> str | None:
        match = ACTION.search(self.text)
        return match.group(0).strip() if match else None


GIVE_UP = "Thought: I do not have the facts I need.\nFinal Answer: unknown"


class StubLLM:
    """A scripted stand-in for a language model.

    On each call it returns the first move whose `requires` are all present in the
    prompt and whose action has not been taken yet. If no move qualifies it gives up,
    which is what a model does when its context lacks the facts the answer needs.
    """

    def __init__(
        self, moves: list[Move], give_up: str = GIVE_UP, repeat: bool = False
    ) -> None:
        self.moves = moves
        self.give_up = give_up
        self.repeat = repeat
        self.calls = 0

    def complete(self, prompt: str) -> str:
        self.calls += 1
        for move in self.moves:
            line = move.action_line
            if line is not None and line in prompt and not self.repeat:
                continue
            if any(cue in prompt for cue in move.forbids):
                continue
            if all(cue in prompt for cue in move.requires):
                return move.text
        return self.give_up

## 4. The loop

Thirty-one lines, blank ones included, and this is the version in the post.
Everything an agent framework wraps is here: build the prompt, read one action, run
one tool, append the observation, repeat.

In [ ]:
ACTION = re.compile(r"^Action:\s*(\w+)\[(.*)\]\s*$", re.MULTILINE)
FINAL = re.compile(r"^Final Answer:\s*(.*)$", re.MULTILINE)


def minimal_react(llm, question, tools, max_steps=8):
    """Reason, act, observe, repeat. This is the whole agent."""
    scratchpad = ""
    for _ in range(max_steps):
        reply = llm(f"{SYSTEM}\nQuestion: {question}\n{scratchpad}")

        answer = FINAL.search(reply)
        if answer:
            return answer.group(1).strip(), scratchpad

        action = ACTION.search(reply)
        if action is None:
            observation = "No action found. Write Action: tool[argument]."
        else:
            name, argument = action.group(1), action.group(2)
            if name not in tools:
                observation = f"Unknown tool {name!r}. Available: {', '.join(tools)}."
            else:
                try:
                    observation = tools[name](argument)
                except Exception as exc:
                    observation = f"{name} failed: {exc}"

        scratchpad += f"{reply.strip()}\nObservation: {observation}\n"
    return None, scratchpad

### The same loop, instrumented

The rest of the notebook uses `run_react`, which is `minimal_react` with three
additions the measurements need: it records the size of every prompt and the parts
of every step, it lets each guard be switched off so the failure modes can be shown,
and it can truncate observations to a budget. The decision logic is unchanged.

In [ ]:
@dataclass
class Step:
    thought: str
    action: str
    observation: str
    prompt_chars: int


@dataclass
class Trace:
    question: str
    steps: list[Step] = field(default_factory=list)
    answer: str | None = None
    outcome: str = "running"
    llm_calls: int = 0
    prompt_chars: list[int] = field(default_factory=list)

    @property
    def solved_as(self) -> str:
        return (self.answer or "").strip()


def run_react(
    llm: StubLLM,
    question: str,
    *,
    max_steps: int | None = 8,
    guard_unknown_tool: bool = True,
    guard_bad_action: bool = True,
    observation_chars: int | None = None,
    truncate: str = "head",
) -> Trace:
    """Run the reason-act-observe loop until the model answers or a guard stops it.

    The four keyword arguments are the guards, each switchable so the post can show
    what the loop does without them. `max_steps=None` removes the step cap entirely.
    """
    trace = Trace(question=question)
    scratchpad = ""

    while max_steps is None or len(trace.steps) < max_steps:
        prompt = build_prompt(question, scratchpad)
        trace.prompt_chars.append(len(prompt))
        reply = llm.complete(prompt)

        final = FINAL.search(reply)
        if final:
            trace.answer = final.group(1).strip()
            trace.outcome = "answered"
            trace.llm_calls = llm.calls
            return trace

        match = ACTION.search(reply)
        if match is None:
            if not guard_bad_action:
                raise ValueError(f"could not parse an action from: {reply!r}")
            observation = (
                "No action found. Write exactly one line of the form "
                "Action: tool[argument], with square brackets."
            )
            tool_name, argument = "-", "-"
        else:
            tool_name, argument = match.group(1), match.group(2)
            tool = TOOLS.get(tool_name)
            if tool is None:
                if not guard_unknown_tool:
                    raise KeyError(tool_name)
                observation = (
                    f"Unknown tool {tool_name!r}. Available tools: "
                    f"{', '.join(TOOLS)}."
                )
            else:
                try:
                    observation = tool(argument)
                except ToolError as exc:
                    if not guard_bad_action:
                        raise
                    observation = str(exc)

        observation = _truncate(observation, observation_chars, truncate)
        thought = reply.split("Action:")[0].replace("Thought:", "").strip()
        trace.steps.append(
            Step(thought, f"{tool_name}[{argument}]", observation, len(prompt))
        )
        scratchpad += f"{reply.strip()}\nObservation: {observation}\n"

    trace.outcome = "step_cap"
    trace.llm_calls = llm.calls
    return trace


def _truncate(text: str, limit: int | None, mode: str) -> str:
    if limit is None or len(text) <= limit:
        return text
    if mode == "head":
        return text[:limit] + "..."
    half = limit // 2
    return text[:half] + " ... " + text[-half:]

## 5. One run, end to end

The question below names a paper by its title, and that title is not an entry. The
first lookup misses and comes back with suggestions, so the second action is one
that could not have been planned from the question alone.

In [ ]:
"""The question set the ReAct post measures on, plus the three failure-mode scripts.

Each task carries the scripted moves for the agent and the plan a planner would
write from the question alone. Seven questions name entries that exist in the
encyclopedia; five name something that does not, so the first search comes back
with suggestions and the agent has to read them before it can continue.
"""


from dataclasses import dataclass, field



@dataclass(frozen=True)
class Task:
    name: str
    kind: str  # "direct" or "redirect"
    question: str
    moves: list[Move]
    plan: list[tuple[str, str]]
    answer: str


def _gap(
    name: str,
    question: str,
    first: tuple[str, str],
    second: tuple[str, str],
    answer: str,
) -> Task:
    """A two-lookup question whose entries can both be found by their own titles."""
    (title_a, year_a), (title_b, year_b) = first, second
    return Task(
        name=name,
        kind="direct",
        question=question,
        moves=[
            Move(f"Thought: I need the year for {title_a}.\nAction: search[{title_a}]"),
            Move(f"Thought: Now the year for {title_b}.\nAction: search[{title_b}]"),
            Move(
                f"Thought: I have both years, so I can subtract.\n"
                f"Action: calculator[{year_b}-{year_a}]",
                requires=(year_a, year_b),
            ),
            Move(
                f"Thought: The gap is {answer} years.\nFinal Answer: {answer}",
                requires=(f"Observation: {answer}\n",),
            ),
        ],
        plan=[("search", title_a), ("search", title_b)],
        answer=answer,
    )


def _redirect(
    name: str,
    question: str,
    phrase: str,
    target: tuple[str, str],
    known: tuple[str, str],
    answer: str,
) -> Task:
    """A question that names something by description rather than by entry title."""
    (title_t, year_t), (title_k, year_k) = target, known
    older, newer = sorted((int(year_t), int(year_k)))
    return Task(
        name=name,
        kind="redirect",
        question=question,
        moves=[
            Move(f"Thought: I will look up {phrase}.\nAction: search[{phrase}]"),
            Move(
                f"Thought: That was not an entry, but {title_t} was suggested.\n"
                f"Action: search[{title_t}]",
                requires=(f"Similar: {title_t}",),
            ),
            Move(f"Thought: Now the year for {title_k}.\nAction: search[{title_k}]"),
            Move(
                f"Thought: I have both years, so I can subtract.\n"
                f"Action: calculator[{newer}-{older}]",
                requires=(year_t, year_k),
            ),
            Move(
                f"Thought: The gap is {answer} years.\nFinal Answer: {answer}",
                requires=(f"Observation: {answer}\n",),
            ),
        ],
        plan=[("search", phrase), ("search", title_k)],
        answer=answer,
    )


TASKS: list[Task] = [
    _gap(
        "lstm-transformer",
        "How many years separate the LSTM paper from the Transformer paper?",
        ("LSTM", "1997"),
        ("Transformer", "2017"),
        "20",
    ),
    _gap(
        "alexnet-transformer",
        "How many years separate AlexNet from the Transformer paper?",
        ("AlexNet", "2012"),
        ("Transformer", "2017"),
        "5",
    ),
    _gap(
        "bert-gpt3",
        "How many years separate BERT from GPT-3?",
        ("BERT", "2018"),
        ("GPT-3", "2020"),
        "2",
    ),
    _gap(
        "word2vec-react",
        "How many years separate Word2Vec from ReAct?",
        ("Word2Vec", "2013"),
        ("ReAct", "2022"),
        "9",
    ),
    _gap(
        "alexnet-resnet",
        "How many years separate AlexNet from ResNet?",
        ("AlexNet", "2012"),
        ("ResNet", "2015"),
        "3",
    ),
    _gap(
        "dropout-bert",
        "How many years separate Dropout from BERT?",
        ("Dropout", "2014"),
        ("BERT", "2018"),
        "4",
    ),
    _gap(
        "adam-gpt3",
        "How many years separate Adam from GPT-3?",
        ("Adam", "2015"),
        ("GPT-3", "2020"),
        "5",
    ),
    _redirect(
        "attention-lstm",
        "How many years separate the LSTM paper from Attention Is All You Need?",
        "Attention Is All You Need",
        ("Transformer", "2017"),
        ("LSTM", "1997"),
        "20",
    ),
    _redirect(
        "residual-alexnet",
        "How many years separate the paper that introduced residual connections "
        "from AlexNet?",
        "residual connections",
        ("ResNet", "2015"),
        ("AlexNet", "2012"),
        "3",
    ),
    _redirect(
        "hochreiter-bert",
        "How many years separate the Hochreiter and Schmidhuber network from BERT?",
        "Hochreiter and Schmidhuber",
        ("LSTM", "1997"),
        ("BERT", "2018"),
        "21",
    ),
    _redirect(
        "embeddings-gpt3",
        "How many years separate the word embeddings paper from GPT-3?",
        "word embeddings",
        ("Word2Vec", "2013"),
        ("GPT-3", "2020"),
        "7",
    ),
    _redirect(
        "adaptive-transformer",
        "How many years separate the adaptive moment optimiser from the Transformer?",
        "adaptive moment optimiser",
        ("Adam", "2015"),
        ("Transformer", "2017"),
        "2",
    ),
]


# A longer trajectory, used to measure how the prompt grows step by step.
EARLIEST_TITLES = [
    ("AlexNet", "2012"),
    ("BERT", "2018"),
    ("Dropout", "2014"),
    ("Adam", "2015"),
    ("GPT-3", "2020"),
    ("ReAct", "2022"),
    ("Word2Vec", "2013"),
    ("ResNet", "2015"),
]

LONG_TASK = Task(
    name="earliest-of-eight",
    kind="direct",
    question=(
        "Which of these was published earliest: AlexNet, BERT, Dropout, Adam, "
        "GPT-3, ReAct, Word2Vec, ResNet?"
    ),
    moves=[
        Move(f"Thought: Checking {title}.\nAction: search[{title}]")
        for title, _ in EARLIEST_TITLES
    ]
    + [
        Move(
            "Thought: 2012 is the earliest of the eight years.\n"
            "Final Answer: AlexNet",
            requires=tuple(year for _, year in EARLIEST_TITLES),
        )
    ],
    plan=[("search", title) for title, _ in EARLIEST_TITLES],
    answer="AlexNet",
)


# ---------------------------------------------------------------- failure modes


@dataclass(frozen=True)
class FailureCase:
    name: str
    label: str
    question: str
    moves: list[Move]
    repeat: bool = False
    guards: dict = field(default_factory=dict)
    answer: str = ""


FAILURES: list[FailureCase] = [
    FailureCase(
        name="no-termination",
        label="never stops",
        question="Which model was published earliest?",
        moves=[
            Move("Thought: I should check the encyclopedia.\nAction: search[AlexNet]")
        ],
        repeat=True,
        guards={"max_steps": None},
        answer="",
    ),
    FailureCase(
        name="unknown-tool",
        label="invents a tool",
        question="In which year was the LSTM published?",
        moves=[
            Move("Thought: I will check Wikipedia.\nAction: wikipedia[LSTM]"),
            Move(
                "Thought: There is no wikipedia tool, so I will use search.\n"
                "Action: search[LSTM]",
                requires=("Unknown tool",),
            ),
            Move(
                "Thought: The entry gives the year.\nFinal Answer: 1997",
                requires=("1997",),
            ),
        ],
        guards={"guard_unknown_tool": False},
        answer="1997",
    ),
    FailureCase(
        name="malformed-action",
        label="malformed action",
        question="What is 2017 minus 1997?",
        moves=[
            Move(
                "Thought: I can work this out with the calculator.\n"
                "Action: calculator(2017-1997)",
                forbids=("No action found",),
            ),
            Move(
                "Thought: The action needs square brackets.\n"
                "Action: calculator[2017-1997]",
                requires=("No action found",),
            ),
            Move(
                "Thought: The calculator returned 20.\nFinal Answer: 20",
                requires=("Observation: 20\n",),
            ),
        ],
        guards={"guard_bad_action": False},
        answer="20",
    ),
]

In [ ]:
task = TASKS[7]
stub = StubLLM(task.moves)

answer, scratchpad = minimal_react(stub.complete, task.question, TOOLS)
print("QUESTION:", task.question, "\n")
print(scratchpad)
print("Final Answer:", answer, f"(expected {task.answer})")

The instrumented loop gives the same run, broken into inspectable steps.

In [ ]:
stub = StubLLM(task.moves)
trace = run_react(stub, task.question, max_steps=8)

print("QUESTION:", task.question, "\n")
for step in trace.steps:
    print(f"Thought: {step.thought}")
    print(f"Action: {step.action}")
    print(f"Observation: {step.observation}\n")
print("Final Answer:", trace.answer)
print(f"({trace.llm_calls} model calls, expected answer {task.answer})")

## 6. Interleaving against planning up front

The alternative is to fix the whole tool sequence before running any of it, then
reason once over everything that came back. `run_plan_execute` does that, and is
given one adaptive step after its plan finishes, which is more than the canonical
version gets.

Seven of the twelve questions name entries that exist under their own titles. The
other five name something by description, so the first lookup misses.

In [ ]:
def run_plan_execute(
    llm: StubLLM,
    question: str,
    plan: list[tuple[str, str]],
    *,
    follow_up_steps: int = 1,
) -> Trace:
    """Fix the whole tool sequence up front, run it, then reason once over the results.

    The `follow_up_steps` allowance is deliberately generous: the canonical
    plan-and-execute baseline gets no adaptive step at all.
    """
    trace = Trace(question=question)
    scratchpad = ""

    for tool_name, argument in plan:
        observation = TOOLS[tool_name](argument)
        scratchpad += (
            f"Thought: following the plan.\nAction: {tool_name}[{argument}]\n"
            f"Observation: {observation}\n"
        )
        trace.steps.append(
            Step("following the plan.", f"{tool_name}[{argument}]", observation, 0)
        )

    for taken in range(follow_up_steps + 1):
        prompt = build_prompt(question, scratchpad)
        trace.prompt_chars.append(len(prompt))
        reply = llm.complete(prompt)

        final = FINAL.search(reply)
        if final:
            trace.answer = final.group(1).strip()
            trace.outcome = "answered"
            trace.llm_calls = llm.calls
            return trace

        match = ACTION.search(reply)
        if match is None or taken == follow_up_steps:
            break
        tool_name, argument = match.group(1), match.group(2)
        try:
            observation = TOOLS[tool_name](argument)
        except (KeyError, ToolError) as exc:
            observation = str(exc)
        scratchpad += f"{reply.strip()}\nObservation: {observation}\n"
        trace.steps.append(
            Step("", f"{tool_name}[{argument}]", observation, len(prompt))
        )

    trace.outcome = "out_of_steps"
    trace.llm_calls = llm.calls
    return trace

In [ ]:
def solved(trace, task):
    return (trace.answer or "").strip() == task.answer

rows = []
for task in TASKS:
    react = run_react(StubLLM(task.moves), task.question, max_steps=8)
    one = run_plan_execute(StubLLM(task.moves), task.question, task.plan,
                           follow_up_steps=1)
    two = run_plan_execute(StubLLM(task.moves), task.question, task.plan,
                           follow_up_steps=2)
    rows.append((task, react, one, two))

print(f"{'question':22} {'kind':9} {'ReAct':>6} {'plan+1':>7} {'plan+2':>7} {'calls':>6}")
for task, react, one, two in rows:
    print(f"{task.name:22} {task.kind:9} "
          f"{('ok' if solved(react, task) else 'FAIL'):>6} "
          f"{('ok' if solved(one, task) else 'FAIL'):>7} "
          f"{('ok' if solved(two, task) else 'FAIL'):>7} "
          f"{react.llm_calls:>6}")

n = len(rows)
print(f"\nReAct          {sum(solved(r[1], r[0]) for r in rows)}/{n}")
print(f"plan then execute, one adaptive step  "
      f"{sum(solved(r[2], r[0]) for r in rows)}/{n}")
print(f"plan then execute, two adaptive steps "
      f"{sum(solved(r[3], r[0]) for r in rows)}/{n}")

The loop answers all twelve. The planner with one adaptive step answers seven, and
the five it misses are exactly the five where the first lookup missed.

Give the planner two adaptive steps and it answers all twelve as well, which is the
more useful result: the gap measures how much adaptation you budgeted before seeing
any data, and a planner with enough adaptive steps has become a ReAct loop with a
redundant plan in front of it.

## 7. What the scratchpad costs

The prompt is rebuilt in full on every call, so the first observation is paid for
once per remaining step. Input grows with the square of the step count while the
visible context grows linearly.

In [ ]:
trace = run_react(StubLLM(LONG_TASK.moves), LONG_TASK.question, max_steps=12)
running = 0
for i, size in enumerate(trace.prompt_chars, start=1):
    running += size
    print(f"call {i:2}  prompt {size:6,} chars   cumulative {running:7,}")

print(f"\nanswer: {trace.answer}")
print(f"largest prompt {trace.prompt_chars[-1]:,} chars, "
      f"total input {running:,} chars, "
      f"{running / trace.prompt_chars[-1]:.1f}x the final prompt")

## 8. Three ways the loop breaks

Every one of these is a real crash or a real runaway in the unguarded loop. The
guard turns each into an observation the model can read and recover from.

In [ ]:
runaway = FAILURES[0]
uncapped = run_react(StubLLM(runaway.moves, repeat=True), runaway.question,
                     max_steps=100)
capped = run_react(StubLLM(runaway.moves, repeat=True), runaway.question,
                   max_steps=6)
print("1. the model never writes Final Answer")
print(f"   no cap    : still looping at step {len(uncapped.steps)}, "
      f"{sum(uncapped.prompt_chars):,} chars of input")
print(f"   max_steps=6: stopped at step {len(capped.steps)} "
      f"({capped.outcome}), {sum(capped.prompt_chars):,} chars")
print(f"   ratio     : {sum(uncapped.prompt_chars) / sum(capped.prompt_chars):.0f}x")

for case in FAILURES[1:]:
    print(f"\n2/3. {case.label}")
    try:
        run_react(StubLLM(case.moves), case.question, max_steps=8, **case.guards)
        print("   without the guard: no error")
    except Exception as exc:
        print(f"   without the guard: {type(exc).__name__}: {exc}")
    fixed = run_react(StubLLM(case.moves), case.question, max_steps=8)
    print(f"   with the guard   : answered {fixed.answer!r} "
          f"in {len(fixed.steps)} steps")

## 9. Truncating observations

Real tool output is long, so you truncate, and truncation is where careful agents
quietly break. Nothing raises an error in the failing runs below: the agent reads a
real observation, finds no year in it, and gives up.

Several entries put the year in the final sentence, which is why keeping half the
budget from each end beats keeping the first N characters.

In [ ]:
print(f"{'budget':>8} {'head only':>10} {'head+tail':>10}")
for limit in [20, 30, 40, 60, 80, 120, 160, 200, None]:
    counts = []
    for mode in ("head", "both"):
        counts.append(sum(
            (run_react(StubLLM(t.moves), t.question, max_steps=8,
                       observation_chars=limit, truncate=mode).answer or "").strip()
            == t.answer
            for t in TASKS
        ))
    print(f"{('none' if limit is None else limit):>8} {counts[0]:>10} {counts[1]:>10}")

## 10. Optional: swap in a real model

Everything above ran against a scripted model, which is why the numbers reproduce
exactly. `run_react` takes the model as an object with a `.complete(prompt)` method,
so a real one drops straight in.

**This is the only cell that needs an API key and the only one that costs money.**
It is not run when the notebook is executed top to bottom.

`stop_sequences` is the load-bearing argument. Without it the model continues past
its own action and writes an `Observation:` of its own invention, which the loop
then parses as though a tool had produced it.

In [ ]:
# pip install anthropic, then set ANTHROPIC_API_KEY, then remove the guard below.
RUN_FOR_REAL = False

if RUN_FOR_REAL:
    import anthropic

    class ClaudeLLM:
        """The same interface as StubLLM, backed by a real model."""

        def __init__(self, model="claude-opus-5"):
            self.client = anthropic.Anthropic()
            self.model = model
            self.calls = 0

        def complete(self, prompt):
            self.calls += 1
            message = self.client.messages.create(
                model=self.model,
                max_tokens=512,
                stop_sequences=["Observation:"],
                messages=[{"role": "user", "content": prompt}],
            )
            return "".join(b.text for b in message.content if b.type == "text")

    llm = ClaudeLLM()
    trace = run_react(llm, TASKS[7].question, max_steps=8)
    for step in trace.steps:
        print(f"Thought: {step.thought}\nAction: {step.action}")
        print(f"Observation: {step.observation}\n")
    print("Final Answer:", trace.answer, f"({llm.calls} model calls)")
else:
    print("Set RUN_FOR_REAL = True and provide an API key to run this cell.")

## Exercises

1. **Add a third tool.** Give the agent a `today[]` tool that returns a fixed date,
   and write a question whose answer needs it. Note how much of the change is in the
   loop (none) and how much is in the prompt and the scripted moves.
2. **Break termination differently.** Script a model that alternates between two
   searches forever. Does `max_steps` still catch it? What would a repeat detector
   have to compare to catch it sooner?
3. **Cap the scratchpad instead of the steps.** Drop the oldest observation once the
   prompt passes 1,200 characters and re-run section 6. Which questions break, and
   is the failure visible from the outside?
4. **Measure the caching win.** Prompt caching bills a cached prefix at a fraction
   of the input rate. Using the per-call sizes from section 7, work out the total
   under caching if every prefix but the last 200 characters is a cache hit.
5. **Swap the text protocol for a schema.** Replace `Action: tool[arg]` with a JSON
   object and re-run section 8's malformed-action case. Which of the three failure
   modes does a schema remove, and which two survive it?